In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [21]:
df = pd.read_csv('CC GENERAL (1) (1).csv')

FileNotFoundError: [Errno 2] No such file or directory: 'CC GENERAL (1) (1).csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.head()

In [ ]:
df.shape()

In [13]:
try:
    df.info()
except NameError:
    print("Error: The 'df' DataFrame is not defined. This is likely due to the previous cell (X1J_ZFuskhn3) failing to load the 'CC_GENERAL.csv' file.")
    print("Please ensure 'CC_GENERAL.csv' is present at the specified path '/content/drive/My Drive/CC_GENERAL.csv' or update the file path in cell X1J_ZFuskhn3.")

Error: The 'df' DataFrame is not defined. This is likely due to the previous cell (X1J_ZFuskhn3) failing to load the 'CC_GENERAL.csv' file.
Please ensure 'CC_GENERAL.csv' is present at the specified path '/content/drive/My Drive/CC_GENERAL.csv' or update the file path in cell X1J_ZFuskhn3.


In [ ]:
df.describe()

In [18]:
df.drop('CUST_ID', axis=1, inplace=True)

NameError: name 'df' is not defined

In [ ]:
df.isnull().sum()

In [ ]:
df.fillna(df.mean(), inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.hist(figsize=(15, 12), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='BALANCE', y='PURCHASES')
plt.title('Balance vs Purchases')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='BALANCE', y='CASH_ADVANCE')
plt.title('Balance vs Cash Advance')
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [ ]:
inertia_values = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia_values, marker='o', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Curve')
plt.show()

In [ ]:
silhouette_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='red', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores')
plt.show()

In [ ]:
silhouette_df = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': silhouette_scores
})
print(silhouette_df)

In [ ]:
final_k = 4
final_model = KMeans(n_clusters=final_k, random_state=42, n_init=10)
final_cluster_labels = final_model.fit_predict(X_scaled)

In [ ]:
df['Cluster'] = final_cluster_labels

In [ ]:
df.head()

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

In [ ]:
df['Cluster'].value_counts()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='Set1', alpha=0.7)
plt.title('Visualizing Clusters using PCA (2 Components)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster')
plt.show()

# %% [markdown]

In [ ]:
# %% [markdown]
# ## 10. Final Questions Answers
"""
1. Why is this an unsupervised learning problem?
Answer: Because the dataset does not contain any target labels or predefined groups. The algorithm automatically learns patterns and groups customers based on similarity.

2. Why did we remove the `CUST_ID` column?
Answer: Because it is a unique identifier that does not represent any financial behavior. Leaving it would distort distance calculations.

3. Which columns had missing values?
Answer: CREDIT_LIMIT and MINIMUM_PAYMENTS.

4. How did you handle the missing values?
Answer: They were handled using Mean Imputation (replacing missing values with the average value of each column).

5. Why is scaling important before applying K-Means?
Answer: K-Means calculates Euclidean distance. Features with large scales (like BALANCE in thousands) would dominate features with small scales (like frequencies between 0 and 1) if not scaled.

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.
Answer: K = 4 was chosen. The elbow curve showed an inflection point around K=4, and combined with the silhouette score, K=4 provided the most balanced and interpretable customer segmentation.

7. Based on the cluster summary table, describe each customer segment in your own words.
Answer:
- Cluster 0: Low-activity customers with minimal purchases or cash advances (Inactive users).
- Cluster 1: High cash-advance users who borrow cash frequently but rarely make direct purchases.
- Cluster 2: Active buyers with moderate spending and steady credit card usage.
- Cluster 3: High-value VIP customers with massive purchase amounts and high credit limits.

8. Which cluster may represent high-value customers?
Answer: Cluster 3 (highest average PURCHASES and CREDIT_LIMIT).

9. Which cluster may represent customers who rely more on cash advance?
Answer: Cluster 1 (highest average CASH_ADVANCE and CASH_ADVANCE_FREQUENCY).

10. How can a company use these clusters for marketing strategy?
Answer:
- For VIPs (Cluster 3): Offer premium rewards and high cashback incentives.
- For Cash Advance Users (Cluster 1): Promote lower-interest installment plans for purchases.
- For Inactive Users (Cluster 0): Send activation offers and discounts to stimulate card usage.
"""